# 06 - NumPy

**Sources:** Summer School *CursP_Numpy_Matplotlib* (Part II: arrays, creation, dtypes, indexing, masks,
fancy indexing, broadcasting, reductions, reshaping, sorting, casting, polynomials) and *BonusSlides*
("speed up your code": vectorising, in-place ops, views vs copies, memory order). Bootcamp BP5 lists
NumPy as the first data-analysis library.

**Modern notes:** use `np.random.default_rng(seed)` (the slides' `np.random.seed` still works but is
legacy); `%pylab` / `from pylab import *` is discouraged - import `numpy as np` explicitly.

In [1]:
import numpy as np
print(np.__version__)

2.5.3


## 1. Why NumPy?
Python lists hold pointers to boxed objects; a NumPy `ndarray` is a **contiguous block of one dtype**.
That gives compact memory and **vectorised** operations executed in C.

In [2]:
import timeit

lst = list(range(1000))
arr = np.arange(1000)
t_list = timeit.timeit(lambda: [i ** 2 for i in lst], number=2000) / 2000
t_arr = timeit.timeit(lambda: arr ** 2, number=2000) / 2000
print(f"list comprehension: {t_list * 1e6:7.1f} us | numpy: {t_arr * 1e6:7.1f} us | speed-up x{t_list / t_arr:.0f}")

list comprehension:    29.6 us | numpy:     0.8 us | speed-up x35


## 2. Creating arrays

In [3]:
a = np.array([0, 1, 2, 3])
print(a, a.ndim, a.shape, len(a), a.dtype, a.size, a.itemsize, a.nbytes)

b = np.array([[0, 1, 2], [3, 4, 5]])          # 2 x 3
print(b, b.ndim, b.shape, len(b))             # len() = size of the first axis

c = np.array([[[1], [2]], [[3], [4]]])
print(c.shape)

[0 1 2 3] 1 (4,) 4 int64 4 8 32
[[0 1 2]
 [3 4 5]] 2 (2, 3) 2
(2, 2, 1)


In [4]:
# We almost never type the elements
print(np.arange(10))                          # 0..n-1
print(np.arange(1, 9, 2))                     # start, stop (exclusive), step
print(np.linspace(0, 1, 6))                   # start, stop (INCLUSIVE), n points
print(np.linspace(0, 1, 5, endpoint=False))
print(np.ones((2, 3)), np.zeros((2, 2)), np.eye(3), sep="\n")
print(np.diag([1, 2, 3, 4]))
print(np.full((2, 2), 7), np.empty((1, 2)).shape, np.zeros_like(b))

[0 1 2 3 4 5 6 7 8 9]
[1 3 5 7]
[0.  0.2 0.4 0.6 0.8 1. ]
[0.  0.2 0.4 0.6 0.8]
[[1. 1. 1.]
 [1. 1. 1.]]
[[0. 0.]
 [0. 0.]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[1 0 0 0]
 [0 2 0 0]
 [0 0 3 0]
 [0 0 0 4]]
[[7 7]
 [7 7]] (1, 2) [[0 0 0]
 [0 0 0]]


In [5]:
# Random numbers: modern Generator API
rng = np.random.default_rng(1234)             # reproducible
print(rng.random(4))                          # uniform [0, 1)
print(rng.normal(size=4))                     # Gaussian (slides: np.random.randn)
print(rng.integers(0, 20, size=8))
print(rng.choice(["a", "b", "c"], size=5))

[0.97669977 0.38019574 0.92324623 0.26169242]
[ 0.86374389  2.91309922 -1.47882336  0.94547297]
[15 19 19  5 11  8  5 12]
['c' 'c' 'b' 'c' 'b']


**Challenges (slides)** - build these without typing every element:
```
[[1 1 1 1]      [[0. 0. 0. 0. 0.]      [[4 3 4 3 4 3]
 [1 1 1 1]       [2. 0. 0. 0. 0.]       [2 1 2 1 2 1]
 [1 1 1 2]       [0. 3. 0. 0. 0.]       [4 3 4 3 4 3]
 [1 6 1 1]]      [0. 0. 4. 0. 0.]       [2 1 2 1 2 1]]
                 [0. 0. 0. 5. 0.]
                 [0. 0. 0. 0. 6.]]
```
Hints: `np.ones`, indexing assignment, `np.diag(v, k=-1)`, `np.tile`.

In [6]:
# Your turn

## 3. Data types
NumPy auto-detects the dtype; the "bigger" type wins in mixed operations; **assignment never changes the dtype**.

In [7]:
print(np.array([1, 2, 3]).dtype, np.array([1., 2, 3]).dtype, np.array([1, 2, 3], dtype=float).dtype)
print(np.array([1 + 2j]).dtype, np.array([True, False]).dtype, np.array(["Bonjour", "Hello"]).dtype)
print(np.array([1, 2, 3]) + 1.5)

x = np.array([1, 2, 3])
x[0] = 1.9                                    # float is truncated!
print(x)

f = np.array([1.7, 1.2, 1.6])
print(f.astype(int), np.round(np.array([1.2, 1.5, 1.6, 2.5, 3.5, 4.5])))   # note: round-half-to-even

small = np.array([100, 120], dtype=np.int8)
print(small + small)                          # silent overflow! (int8 max is 127)
print(np.iinfo(np.int8), np.finfo(np.float64).eps)

int64 float64 float64
complex128 bool <U7
[2.5 3.5 4.5]
[1 2 3]
[1 1 1] [1. 2. 2. 2. 4. 4.]
[-56 -16]
Machine parameters for int8
---------------------------------------------------------------
min = -128
max = 127
---------------------------------------------------------------
 2.220446049250313e-16


## 4. Indexing and slicing
Same as Python sequences, with one index per axis: `a[row, col]`.

In [8]:
a = np.arange(10)
print(a[0], a[2], a[-1], a[2:9:3], a[:4], a[::2], a[3:], a[::-1])

m = np.diag(np.arange(3))
m[2, 1] = 10
print(m, m[1], m[1, 1], sep="\n")

0 2 9 [2 5 8] [0 1 2 3] [0 2 4 6 8] [3 4 5 6 7 8 9] [9 8 7 6 5 4 3 2 1 0]
[[ 0  0  0]
 [ 0  1  0]
 [ 0 10  2]]
[0 1 0]
1


In [9]:
# The classic 6x6 table from the slides
a = np.add.outer(10 * np.arange(6), np.arange(6))      # a[i, j] = 10 i + j
print(a)
print(a[4:, 4:], a[:, 2], a[2::2, ::2], a[0, 3:5], sep="\n")

[[ 0  1  2  3  4  5]
 [10 11 12 13 14 15]
 [20 21 22 23 24 25]
 [30 31 32 33 34 35]
 [40 41 42 43 44 45]
 [50 51 52 53 54 55]]
[[44 45]
 [54 55]]
[ 2 12 22 32 42 52]
[[20 22 24]
 [40 42 44]]
[3 4]


### Views vs copies (important!)
Basic slicing returns a **view** (shares memory). Fancy / boolean indexing returns a **copy**.

In [10]:
a = np.arange(10)
view = a[::2]
view[0] = 99
print(a)                       # a changed!
print(np.shares_memory(a, view))

copy = a[::2].copy()
copy[0] = -1
print(a[0], copy[0])

fancy = a[[1, 3]]              # fancy indexing -> copy
fancy[0] = 1000
print(a[1])

[99  1  2  3  4  5  6  7  8  9]
True
99 -1
1


### Boolean masks

In [11]:
rng = np.random.default_rng(3)
a = rng.integers(0, 20, 15)
print(a)
mask = a % 3 == 0
print(mask)
print(a[mask], a[a % 3 == 0])             # extract
a[a % 3 == 0] = -1                        # assign through a mask
print(a)

# combine conditions with & | ~ and parentheses (NOT `and` / `or`)
b = np.arange(20)
print(b[(b > 5) & (b % 2 == 0)], b[~(b < 15)])
print(np.where(b % 5 == 0, "div5", "-")[:6], np.nonzero(b > 17))

[16  1  3  4  3 16 17 11  0  1  6  8 12  9  5]
[False False  True False  True False False False  True False  True False
  True  True False]
[ 3  3  0  6 12  9] [ 3  3  0  6 12  9]
[16  1 -1  4 -1 16 17 11 -1  1 -1  8 -1 -1  5]
[ 6  8 10 12 14 16 18] [15 16 17 18 19]
['div5' '-' '-' '-' '-' 'div5'] (array([18, 19]),)


### Fancy indexing and `np.newaxis`

In [12]:
a = np.arange(10)
print(a[[2, 3, 2, 4, 2]])                 # repeated indices allowed
a[[9, 7]] = -10
print(a)
idx = np.array([[3, 4], [9, 7]])
print(np.arange(10)[idx])                 # result has the shape of the index array

t = np.add.outer(10 * np.arange(6), np.arange(6))
print(t[(0, 1, 2, 3, 4), (1, 2, 3, 4, 5)])           # picks the diagonal one above the main
print(t[3:, [0, 2, 5]])
print(t[np.array([1, 0, 1, 0, 0, 1], dtype=bool), 2])

z = np.array([1, 2, 3])
print(z[:, np.newaxis], z[np.newaxis, :], z.reshape(-1, 1).shape, sep="\n")

[2 3 2 4 2]
[  0   1   2   3   4   5   6 -10   8 -10]
[[3 4]
 [9 7]]
[ 1 12 23 34 45]
[[30 32 35]
 [40 42 45]
 [50 52 55]]
[ 2 22 52]
[[1]
 [2]
 [3]]
[[1 2 3]]
(3, 1)


**Challenge (5 min):** construct an array containing the prime numbers between 1 and 100
(hint: start from `np.arange(2, 101)`, a boolean mask and a Sieve of Eratosthenes - see Codility lesson 11).

**Challenge:** build `[[1,6,11],[2,7,12],[3,8,13],[4,9,14],[5,10,15]]` without typing it
(`reshape`, `.T` or `arange` + broadcasting), then extract its 2nd and 4th rows.

In [13]:
# Your turn

## 5. Numerical operations: element-wise, reductions

In [14]:
a = np.array([1, 2, 3, 4])
b = np.ones(4) + 1
print(a + 1, 2 ** a, a - b, a * b, np.sqrt(a), np.exp(-a), np.sin(np.pi * a / 2).round(3))
print((2 ** (np.arange(5) + 1)) - np.arange(5))

c = np.ones((3, 3))
print(c * c)                         # element-wise, NOT matrix product
print(c @ c)                         # matrix product (same as c.dot(c))

[2 3 4 5] [ 2  4  8 16] [-1.  0.  1.  2.] [2. 4. 6. 8.] [1.         1.41421356 1.73205081 2.        ] [0.36787944 0.13533528 0.04978707 0.01831564] [ 1.  0. -1. -0.]
[ 2  3  6 13 28]
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]
[[3. 3. 3.]
 [3. 3. 3.]
 [3. 3. 3.]]


In [15]:
# Comparisons and logic
a, b = np.array([1, 2, 3, 4]), np.array([4, 2, 2, 4])
print(a == b, a > b, np.array_equal(a, b), np.allclose(a, a + 1e-12))
p, q = np.array([1, 1, 0, 0], bool), np.array([1, 0, 1, 0], bool)
print(np.logical_or(p, q), np.logical_and(p, q), np.logical_not(p))

[False  True False  True] [False False  True False] False True
[ True  True  True False] [ True False False False] [False False  True  True]


In [16]:
# Reductions: sum/mean/std/min/max, optionally along an axis
x = np.array([[1, 1], [2, 2]])
print(x.sum(), x.sum(axis=0), x.sum(axis=1))      # axis=0 collapses rows -> per-column result
y = np.array([[1, 2, 3], [5, 6, 1]])
print(y.mean(), np.median(y, axis=-1), y.std(), y.var(ddof=1), y.min(axis=0), y.argmax(), y.cumsum(axis=1))
print(np.all([True, True, False]), np.any([True, True, False]))
print(np.any(np.zeros((100, 100)) != 0), np.all(np.zeros((3, 3)) == 0))

6 [3 3] [2 4]
3.0 [2. 5.] 1.9148542155126762 4.4 [1 2 1] 4 [[ 1  3  6]
 [ 5 11 12]]
False True
False True


Reading `axis`: it is the dimension that **disappears**. For shape `(rows, cols)`: `axis=0` -> one value per
column; `axis=1` -> one value per row.

## 6. Broadcasting
Arrays of different shapes are combined by *virtually* stretching size-1 (or missing leading) dimensions.
Rule: compare shapes **from the right**; each pair must be equal, or one of them 1.

In [17]:
a = np.arange(3)
print(a + 10)                                       # scalar
print(np.ones((3, 3)) + np.array([0, 1, 2]))        # (3,3) + (3,)   -> row added to every row
print(np.arange(3)[:, np.newaxis] + np.arange(3))   # (3,1) + (3,)   -> (3,3) table
try:
    np.arange(4) + np.array([1, 2])
except ValueError as err:
    print("ValueError:", err)

[10 11 12]
[[1. 2. 3.]
 [1. 2. 3.]
 [1. 2. 3.]]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
ValueError: operands could not be broadcast together with shapes (4,) (2,) 


In [18]:
# Example from the slides: pairwise distances between mileposts
mileposts = np.array([0, 198, 303, 736, 871, 1175, 1475, 1544, 1913, 2448])
distance_array = np.abs(mileposts - mileposts[:, np.newaxis])
print(distance_array[:4, :6])

# Another classic: normalise columns
data = np.random.default_rng(0).normal(5, 2, size=(100, 3))
z = (data - data.mean(axis=0)) / data.std(axis=0)
print(z.mean(axis=0).round(10), z.std(axis=0).round(10))

[[   0  198  303  736  871 1175]
 [ 198    0  105  538  673  977]
 [ 303  105    0  433  568  872]
 [ 736  538  433    0  135  439]]
[-0.  0.  0.] [1. 1. 1.]


## 7. Shape manipulation, sorting

In [19]:
a = np.array([[1, 2, 3], [4, 5, 6]])
print(a.ravel(), a.T, a.T.ravel(), sep="\n")
print(a.reshape(3, 2), a.reshape(-1), a.reshape(1, -1).shape)
print(np.concatenate([a, a], axis=0).shape, np.vstack([a, a]).shape, np.hstack([a, a]).shape,
      np.stack([a, a]).shape, np.split(np.arange(6), 3))
print(np.resize(np.arange(3), (2, 4)), np.tile(np.array([[4, 3], [2, 1]]), (2, 3)).shape)

[1 2 3 4 5 6]
[[1 4]
 [2 5]
 [3 6]]
[1 4 2 5 3 6]
[[1 2]
 [3 4]
 [5 6]] [1 2 3 4 5 6] (1, 6)
(4, 3) (4, 3) (2, 6) (2, 2, 3) [array([0, 1]), array([2, 3]), array([4, 5])]
[[0 1 2 0]
 [1 2 0 1]] (4, 6)


In [20]:
a = np.array([[4, 3, 5], [1, 2, 1]])
print(np.sort(a, axis=1))          # sorted copy along an axis
a.sort(axis=1)                     # in place
print(a)
v = np.array([4, 3, 1, 2])
j = np.argsort(v)
print(j, v[j], np.argmax(v), np.argmin(v))
print(np.unique([3, 1, 3, 2, 1], return_counts=True))
print(np.searchsorted([1, 3, 5, 7], [0, 3, 4, 8]))     # binary search (Codility lesson 14)

[[3 4 5]
 [1 1 2]]
[[3 4 5]
 [1 1 2]]
[2 3 1 0] [1 2 3 4] 0 2
(array([1, 2, 3]), array([2, 1, 2]))
[0 1 2 4]


## 8. Polynomials, linear algebra and simple fitting

In [21]:
p = np.poly1d([3, 2, -1])                # 3x^2 + 2x - 1
print(p(0), p.roots, p.order, p.deriv(), p(np.array([0, 1, 2])))

rng = np.random.default_rng(1)
x = np.linspace(0, 1, 20)
y = np.cos(x) + 0.3 * rng.random(20)
fit = np.poly1d(np.polyfit(x, y, 3))     # cubic least-squares fit
print(fit.coeffs.round(3), float(np.mean((fit(x) - y) ** 2)))

# New code should use the np.polynomial.Polynomial class
from numpy.polynomial import Polynomial
poly = Polynomial.fit(x, y, 3)
print(poly(0.5).round(4), fit(0.5).round(4))

-1 [-1.          0.33333333] 2  
6 x + 2 [-1  4 15]
[-0.257 -0.129 -0.204  1.19 ] 0.005224977384629894
1.0242 1.0242


In [22]:
A = np.array([[3, 2, 0], [1, -1, 0], [0, 5, 1]], dtype=float)
bvec = np.array([2, 4, -1], dtype=float)
sol = np.linalg.solve(A, bvec)                 # solve A x = b   (don't compute inv(A) @ b)
print(sol, np.allclose(A @ sol, bvec))
print(np.linalg.det(A), np.linalg.inv(A).round(3), np.linalg.eigvals(A).round(3), sep="\n")
U, S, Vt = np.linalg.svd(A)
print(S.round(3), np.allclose(U @ np.diag(S) @ Vt, A))

[ 2. -2.  9.] True
-4.999999999999999
[[ 0.2  0.4  0. ]
 [ 0.2 -0.6  0. ]
 [-1.   3.   1. ]]
[ 1.   +0.j -1.449+0.j  3.449+0.j]
[5.652 2.994 0.295] True


## 9. Performance tips (bonus slides: "speed up your code")
1. **Vectorise**: replace Python loops by array expressions / masks / fancy indexing.
2. **Broadcast** on the smallest arrays before combining.
3. **In-place operations** (`a *= 0`) avoid temporaries.
4. **Views, not copies** - copying big arrays costs as much as arithmetic.
5. **Memory order**: NumPy is row-major (C order); iterate / reduce along the *contiguous* axis.

In [23]:
import timeit
n = 2_000_000
a = np.random.default_rng(0).random(n)

def loop_sum_sq(a):
    total = 0.0
    for v in a[:100_000]:                       # (only 100k: Python loops are slow)
        total += v * v
    return total

t_loop = timeit.timeit(lambda: loop_sum_sq(a), number=3) / 3
t_vec = timeit.timeit(lambda: np.dot(a[:100_000], a[:100_000]), number=3) / 3
print(f"python loop {t_loop * 1e3:.2f} ms vs vectorised {t_vec * 1e3:.3f} ms")

python loop 8.94 ms vs vectorised 0.144 ms


In [24]:
b = np.zeros(10_000_000)
t1 = timeit.timeit(lambda: 0 * b, number=5) / 5     # allocates a new array
def inplace():
    b[...] *= 0                                      # reuses memory
t2 = timeit.timeit(inplace, number=5) / 5
print(f"new array {t1 * 1e3:.1f} ms | in place {t2 * 1e3:.1f} ms")

new array 39.5 ms | in place 11.5 ms


In [25]:
c = np.zeros((4000, 4000))                           # C order: rows are contiguous
print(c.strides, c.flags["C_CONTIGUOUS"])
t_ax0 = timeit.timeit(lambda: c.sum(axis=0), number=3) / 3
t_ax1 = timeit.timeit(lambda: c.sum(axis=1), number=3) / 3
print(f"sum(axis=0): {t_ax0 * 1e3:.1f} ms | sum(axis=1): {t_ax1 * 1e3:.1f} ms")
print("F-order array is the opposite:", np.asfortranarray(c).strides)

(32000, 8) True
sum(axis=0): 24.9 ms | sum(axis=1): 17.2 ms


F-order array is the opposite: (8, 32000)


## 10. Saving and loading

In [26]:
import tempfile, os
tmpdir = tempfile.mkdtemp()
arr = np.arange(12).reshape(3, 4)
np.save(os.path.join(tmpdir, "a.npy"), arr)                       # binary, exact
np.savetxt(os.path.join(tmpdir, "a.csv"), arr, delimiter=",", fmt="%d")   # text
print(np.load(os.path.join(tmpdir, "a.npy")).shape, np.loadtxt(os.path.join(tmpdir, "a.csv"), delimiter=",").shape)
np.savez(os.path.join(tmpdir, "bundle.npz"), a=arr, b=arr.T)
print(list(np.load(os.path.join(tmpdir, "bundle.npz")).keys()))
import shutil; shutil.rmtree(tmpdir)

(3, 4) (3, 4)
['a', 'b']


## Exercises

**A. Moving average** of a 1-D array with window `k` - first with a loop, then vectorised with `np.convolve`
or `np.cumsum` (relate it to *prefix sums*, Codility lesson 5).

**B. Pairwise Euclidean distances** between the rows of an `(n, d)` matrix without loops, using broadcasting;
verify against `scipy.spatial.distance.cdist`.

**C. Game of Life step**: given a 2-D 0/1 array, compute the next generation using slicing/roll only.

**D. Standardise a data matrix** and compute its covariance matrix by hand; compare with `np.cov`.

**E. Monte-Carlo estimate of pi** with 10 million random points - time the vectorised version.

In [27]:
# Your turn